# Memory Update Decisions in 3D Reconstruction

**One-shot pipeline**: downloads 6 real 3DGS scenes, measures coupling (kappa),
trains Q-network, runs CEM planner, computes baselines (random, frequency, heuristic),
generates all figures (k heatmaps, MSE curves), runs deep ablation, and outputs a LaTeX table.

All CPU-safe (no CUDA). Run all cells in order.


In [ ]:
# Cell 1 - Imports + configuration
import numpy as np, torch, torch.nn as nn, torch.optim as optim, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, warnings, os, requests, json, time, textwrap
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
DEVICE = 'cpu'

SCENES = [
    ('bicycle', 'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/bicycle/bicycle-7k-mini.splat'),
    ('bonsai',  'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/bonsai/bonsai-7k-mini.splat'),
    ('counter', 'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/counter/counter-7k.splat'),
    ('garden',  'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/garden/garden-7k.splat'),
    ('room',    'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/room/room-7k.splat'),
    ('kitchen', 'https://huggingface.co/datasets/dylanebert/3dgs/resolve/main/kitchen/kitchen-7k.splat'),
]
OUTPUT = os.path.join(os.getcwd(), '..')  # parent of kaggle/working
os.makedirs(os.path.join(OUTPUT, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT, 'results'), exist_ok=True)
print(f'{len(SCENES)} scenes, output to {OUTPUT}')


In [ ]:
# Cell 2 - Core classes and functions
class VoxelWorld3D:
    def __init__(self, N=80, G=16, T=120, pos_raw=None):
        self.N, self.G, self.T = N, G, T
        if pos_raw is not None: self._from_real(pos_raw)
        else: self._gen_synthetic()
        self._gen_motion()
        self._gen_observations()
        self._render_all()

    def _from_real(self, pos_v):
        N, G = self.N, self.G
        if len(pos_v)==0: self.true_pos0=np.random.uniform(.5,G-.5,(N,3)); self.sig=np.random.uniform(2.5,5,N); self.amp=np.ones(N); self.cid=np.random.randint(0,max(2,N//10),N); return
        center = pos_v[np.random.choice(len(pos_v))] if len(pos_v)>0 else np.zeros(3)
        dists = np.linalg.norm(pos_v - center, axis=1)
        r = np.sort(dists)[min(3000, len(pos_v)-1)]
        pos_r = pos_v[dists <= r]
        if len(pos_r) > 3000:
            pos_r = pos_r[np.random.choice(len(pos_r), 3000, replace=False)]
        kmeans = KMeans(n_clusters=N, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(pos_r[:, :3])
        centroids = kmeans.cluster_centers_
        sig = np.zeros(N)
        for i in range(N):
            mem = pos_r[labels == i]
            sig[i] = np.mean(np.linalg.norm(mem[:,:3]-centroids[i],axis=1))*3.0 if len(mem)>1 else 0.1
        pmin, pmax = centroids.min(0), centroids.max(0)
        pr = np.maximum(pmax - pmin, 0.1)
        p_norm = (centroids - pmin) / pr * (G - 1) + 0.5
        sig_norm = sig / pr.mean() * G * 1.0
        self.sig = np.clip(sig_norm, 0.5, G * 0.4)
        self.amp = np.clip(np.ones(N)*1.2, 0.3, 2.0)
        self.cid = np.random.randint(0, max(2, N//10), N)
        self.true_pos0 = p_norm

    def _gen_synthetic(self):
        N, G = self.N, self.G
        self.sig = np.random.uniform(2.5, 5.0, N)
        self.amp = np.random.uniform(0.5, 1.5, N)
        nc = N // 6
        c = np.random.uniform(G*.3, G*.7, (nc,3))
        self.cid = np.random.randint(0, nc, N)
        p = np.zeros((N,3))
        for i in range(N): p[i] = c[self.cid[i]] + np.random.normal(0, self.sig[i]*.5, 3)
        self.true_pos0 = np.clip(p, .5, G-.5)

    def _gen_motion(self):
        N, T, G = self.N, self.T, self.G
        c = np.zeros((N,3))
        for ci in np.unique(self.cid):
            ii = np.where(self.cid == ci)[0]
            c[ii] = self.true_pos0[ii].mean(0)
        t = np.zeros((T,N,3)); t[0] = self.true_pos0
        for k in range(1,T):
            t[k] = t[k-1] + np.random.normal(0,.3,(N,3)) + .05*(c-t[k-1])
            t[k] = np.clip(t[k], .5, G-.5)
        self.true = t; self.cluster_c = c

    def _gen_observations(self):
        N, T = self.N, self.T
        ns = np.random.uniform(.4, 1.5, N)
        self.obs = self.true + np.random.normal(0,1,(T,N,3))*ns[None,:,None]
        self.obs = np.clip(self.obs, 0, self.G-.01)
        d = np.linalg.norm(self.true - self.cluster_c[None], axis=-1)
        self.conf = np.clip(1 - .3 * d / np.maximum(self.sig[None], 1), .2, 1.)

    def _r(self, x):
        G=self.G; a,b,c=np.meshgrid(*[range(G)]*3,indexing='ij'); v=np.zeros((G,G,G))
        for e in range(self.N): v+=self.amp[e]*np.exp(-((a-x[e,0])**2+(b-x[e,1])**2+(c-x[e,2])**2)/(2*self.sig[e]**2))
        return v

    def _render_all(self):
        self.vg = np.zeros((self.T,self.G,self.G,self.G))
        self.vo = np.zeros((self.T,self.G,self.G,self.G))
        for k in range(self.T): self.vg[k]=self._r(self.true[k]); self.vo[k]=self._r(self.obs[k])

    def err(self,a,b): return np.mean((a-b)**2)
    def base(self,t): return self.err(self.vo[t],self.vg[t])
    def upd(self,idxs,t):
        p=self.obs[t].copy(); p[list(idxs)]=self.true[t][list(idxs)]
        return self.err(self._r(p),self.vg[t]),

def kappa(w, t=0):
    b=w.base(t); K=np.zeros((w.N,w.N))
    for i in range(w.N):
        di=b-w.upd([i],t)[0]
        for j in range(i+1,w.N):
            dj=b-w.upd([j],t)[0]; dij=b-w.upd([i,j],t)[0]
            K[i,j]=K[j,i]=dij-(di+dj)
    return K

def greedy(w, K, t):
    sel=[]; ce=w.base(t)
    for _ in range(K):
        bg,bi=-np.inf,-1
        for i in range(w.N):
            if i in sel: continue
            p=w.obs[t].copy(); p[i]=w.true[t][i]; ne=w.err(w._r(p),w.vg[t])
            if ce-ne>bg: bg,bi=ce-ne,i
        sel.append(bi)
        p=w.obs[t].copy(); p[list(sel)]=w.true[t][list(sel)]; ce=w.err(w._r(p),w.vg[t])
    return sel,ce

def bf_opt(w, K, t, ms=3000):
    a=list(combinations(range(w.N),K))
    if len(a)>ms: a=[a[i] for i in np.random.choice(len(a),ms,0)]
    be,bs=np.inf,None
    for s in a:
        e=w.upd(list(s),t)[0]
        if e<be: be,bs=e,s
    return list(bs),be

def overlap_iou(w):
    N,G=w.N,w.G; a,b,c=np.meshgrid(*[range(G)]*3,indexing='ij'); O=np.zeros((N,N))
    for i in range(N):
        gi=w.amp[i]*np.exp(-((a-w.true[0,i,0])**2+(b-w.true[0,i,1])**2+(c-w.true[0,i,2])**2)/(2*w.sig[i]**2))
        for j in range(i+1,N):
            gj=w.amp[j]*np.exp(-((a-w.true[0,j,0])**2+(b-w.true[0,j,1])**2+(c-w.true[0,j,2])**2)/(2*w.sig[j]**2))
            O[i,j]=O[j,i]=np.sum(np.minimum(gi,gj))/max(np.sum(np.maximum(gi,gj)),1e-10)
    return O

def feats(w,t,e,o):
    G=w.G; p=w.obs[t,e]/G
    m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
    u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G
    cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
    nv=np.sum(o[e]>.05)/(w.N-1)
    cl=np.zeros(5); cl[w.cid[e]%5]=1.
    return np.r_[p,m,[u,cc,ss,a,nv],cl]

def train_q(X, y, verbose=True, dropout=True):
    Xtr,Xva,ytr,yva=train_test_split(X,y,test_size=.2,random_state=42)
    sx=StandardScaler().fit(Xtr); Xtr_s=sx.transform(Xtr); Xva_s=sx.transform(Xva)
    sy=StandardScaler().fit(ytr.reshape(-1,1))
    ytr_n=sy.transform(ytr.reshape(-1,1)).ravel(); yva_n=sy.transform(yva.reshape(-1,1)).ravel()
    D = X.shape[1]
    if dropout:
        mdl=nn.Sequential(nn.Linear(D,64),nn.ReLU(),nn.Dropout(0.1),
                          nn.Linear(64,32),nn.ReLU(),nn.Dropout(0.1),nn.Linear(32,1)).to(DEVICE)
    else:
        mdl=nn.Sequential(nn.Linear(D,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1)).to(DEVICE)
    opt=optim.AdamW(mdl.parameters(),lr=1e-3,weight_decay=0.01)
    loss_fn=nn.MSELoss()
    loader=DataLoader(TensorDataset(torch.FloatTensor(Xtr_s),torch.FloatTensor(ytr_n)),64,shuffle=True)
    val_t,val_y=torch.FloatTensor(Xva_s),torch.FloatTensor(yva_n)
    best_v=float('inf')
    for ep in range(200):
        mdl.train(); el=0
        for bx,by in loader:
            opt.zero_grad(); l=loss_fn(mdl(bx).squeeze(),by); l.backward(); opt.step(); el+=l.item()
        mdl.eval()
        with torch.no_grad(): vl=loss_fn(mdl(val_t).squeeze(),val_y).item()
        if vl<best_v: best_v=vl
        if verbose and (ep+1)%100==0: print(f'  Ep {ep+1}: tr={el/len(loader):.4f} val={vl:.4f}')
    if verbose: print(f'Best val loss: {best_v:.4f} (baseline=1.0)')
    return mdl, sx, sy, best_v

def h_score(w,t,o):
    s=np.zeros(w.N)
    for e in range(w.N):
        m=np.linalg.norm(w.obs[t,e]-w.obs[t-1,e]) if t>0 else 0
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])
        s[e]=m+u+(1-w.conf[t,e])
    return s

def l_score(w,t,o,mdl_,sx_,sy_):
    s=np.zeros(w.N)
    for e in range(w.N):
        f=sx_.transform(feats(w,t,e,o).reshape(1,-1))
        with torch.no_grad():
            q=mdl_(torch.FloatTensor(f)).cpu().numpy().reshape(-1,1)
            s[e]=sy_.inverse_transform(q).item()
    return s

def cem_planner(w, t, K, l_score_fn, ov, n_iters=4, n_samples=80, n_elite=16):
    N = w.N
    q = l_score_fn(w, t, ov)
    q = np.maximum(q, 0) + 1e-10
    probs = q / q.sum()
    best_set, best_s = None, float('inf')
    for it in range(n_iters):
        sets = [np.random.choice(N, K, replace=False, p=probs) for _ in range(n_samples)]
        ss = np.array([w.upd(list(s), t)[0] for s in sets])
        idx = np.argsort(ss)
        new_p = np.zeros(N)
        for s in [sets[i] for i in idx[:n_elite]]: new_p[list(s)] += 1
        new_p /= new_p.sum()
        probs = 0.7 * new_p + 0.3 * probs
        if ss[idx[0]] < best_s: best_s, best_set = ss[idx[0]], sets[idx[0]]
    return list(best_set), best_s

def bootstrap_ci(vals, n_boot=1000, ci=0.95):
    means = [np.mean(np.random.choice(vals, len(vals), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(means, (1-ci)/2*100)
    hi = np.percentile(means, (1+ci)/2*100)
    return np.mean(vals), lo, hi

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.ndarray,)): return obj.tolist()
        if isinstance(obj, (np.bool_,)): return bool(obj)
        return super().default(obj)

print('All classes and functions defined.')


In [ ]:
# Cell 3 - Scene pipeline
def download_scene(url, timeout=300):
    r = requests.get(url, timeout=timeout)
    raw = np.frombuffer(r.content, dtype=np.uint8)
    offset = 0
    if raw[:4].tobytes() == b'splt':
        offset = 8 + int.from_bytes(raw[4:8], 'little')
    gs = raw[offset:]
    n_g = len(gs) // 32
    gs = gs[:n_g*32].reshape(n_g, 32)
    pos_all = gs[:, 0:12].view('<f4').reshape(n_g, 3)
    scale_all = gs[:, 16:28].view('<f4').reshape(n_g, 3)
    alpha_all = gs[:, 15].astype(np.float32) / 255.0
    valid = np.all(np.isfinite(scale_all), axis=1) & (scale_all.max(1) < 100) & (scale_all.min(1) > 1e-10)
    return pos_all[valid], scale_all[valid], alpha_all[valid], len(gs)

def run_scene(name, url, output_dir, N=80, G=16, T=120, n_eval=30, K_max=8, cem_iters=4, cem_samples=80):
    print(f'\n{"="*60}')
    print(f'Scene: {name}')
    print(f'{"="*60}')
    t0 = time.time()

    pos_v, _, _, n_total = download_scene(url)
    print(f'  [{(time.time()-t0):.0f}s] {n_total} Gaussians, {len(pos_v)} valid')

    w = VoxelWorld3D(N=N, G=G, T=T, pos_raw=pos_v)
    print(f'  [{(time.time()-t0):.0f}s] {w.N} meta-entities')

    # Kappa
    K = kappa(w); kf = K[np.triu_indices_from(K, k=1)]
    kappa_mean = float(np.mean(np.abs(kf)))
    pairs_coupled = int(np.sum(np.abs(kf) > 0.001))
    total_pairs = len(kf)
    pct_coupled = 100 * pairs_coupled / total_pairs
    gate_pass = kappa_mean > 0.0005
    print(f'  [{(time.time()-t0):.0f}s] |k|={kappa_mean:.6f} coupled={pairs_coupled}/{total_pairs} ({pct_coupled:.1f}%) gate={gate_pass}')

    # Save kappa heatmap
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(K, cmap='RdBu_r', vmin=-np.percentile(np.abs(kf),95), vmax=np.percentile(np.abs(kf),95))
    plt.colorbar(im, ax=ax, label='k (coupling)')
    ax.set_title(f'{name}: |k|={kappa_mean:.4f} ({pct_coupled:.0f}% pairs coupled)')
    ax.set_xlabel('Entity'); ax.set_ylabel('Entity')
    fig.savefig(os.path.join(output_dir, 'figures', f'kappa_{name}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  [{(time.time()-t0):.0f}s] Kappa heatmap saved')

    # Regret
    regs = []
    for K_reg in [2,3,4]:
        rk = []
        for _ in range(10):
            t = np.random.randint(0, w.T)
            _, eg = greedy(w, K_reg, t)
            _, eo = bf_opt(w, K_reg, t)
            rk.append(eg - eo)
        regs.append(float(np.mean(rk)))
    gate_regret = regs[0] > 0
    print(f'  [{(time.time()-t0):.0f}s] Regret(2)={regs[0]:.6f} regret(3)={regs[1]:.6f} regret(4)={regs[2]:.6f}')

    overall_gate = gate_pass and gate_regret
    print(f'  [{(time.time()-t0):.0f}s] Gate: coupling={gate_pass} regret={gate_regret} overall={overall_gate}')

    # Overlap + dataset
    ov = overlap_iou(w)
    X, y = [], []
    for t in range(w.T):
        b = w.base(t)
        for e in range(w.N): X.append(feats(w, t, e, ov)); y.append(b - w.upd([e], t)[0])
    X, y = np.array(X), np.array(y)
    print(f'  [{(time.time()-t0):.0f}s] Dataset: {X.shape[0]} x {X.shape[1]} feats')

    # Train Q
    mdl, sx, sy, val_loss = train_q(X, y, verbose=False)
    print(f'  [{(time.time()-t0):.0f}s] Q trained: val_loss={val_loss:.4f}')

    # Evaluate all methods
    ft = np.random.choice(w.T, n_eval, replace=False)
    Kr = list(range(1, K_max + 1))
    rh, rl, rc, ro, rr, rf = [], [], [], [], [], []
    l_score_curried = lambda w,t,o: l_score(w,t,o,mdl,sx,sy)
    all_errors = {}
    for K_v in Kr:
        eh, el, ec, eo, er, ef = [], [], [], [], [], []
        for t in ft:
            # Heuristic
            sh = h_score(w, t, ov); eh.append(w.upd(list(np.argsort(sh)[::-1][:K_v]), t)[0])
            # Q-learned
            sl = l_score_curried(w, t, ov); el.append(w.upd(list(np.argsort(sl)[::-1][:K_v]), t)[0])
            # CEM
            _, ec_val = cem_planner(w, t, K_v, l_score_curried, ov, cem_iters, cem_samples); ec.append(ec_val)
            # Random baseline
            er.append(w.upd(list(np.random.choice(w.N, K_v, replace=False)), t)[0])
            # Frequency baseline (most motion)
            motions = np.array([np.linalg.norm(w.obs[t,e]-w.obs[t-1,e]) if t>0 else 0 for e in range(w.N)])
            ef.append(w.upd(list(np.argsort(motions)[::-1][:K_v]), t)[0])
            # Oracle (K<=4)
            if K_v <= 4: eo.append(bf_opt(w, K_v, t, 2000)[1])
        rh.append(np.mean(eh)); rl.append(np.mean(el)); rc.append(np.mean(ec))
        rr.append(np.mean(er)); rf.append(np.mean(ef))
        all_errors[str(K_v)] = {'heuristic':eh, 'q_learned':el, 'cem':ec, 'random':er, 'freq':ef}
        if eo: ro.append(np.mean(eo))
        print(f'  [{(time.time()-t0):.0f}s] K={K_v}: H={np.mean(eh):.4f} Q={np.mean(el):.4f} C={np.mean(ec):.4f} R={np.mean(er):.4f} F={np.mean(ef):.4f}' + (f' O={np.mean(eo):.4f}' if eo else ''))

    # Bootstrap CIs
    cis = {}
    for k in Kr:
        cis[str(k)] = {}
        for method in ['heuristic','q_learned','cem','random','freq']:
            vals = all_errors[str(k)][method]
            m, lo, hi = bootstrap_ci(vals)
            cis[str(k)][method] = {'mean':float(m), 'ci_lo':float(lo), 'ci_hi':float(hi)}

    # Save MSE curve figure
    fig, ax = plt.subplots(figsize=(7,4))
    methods = [('Heuristic',rh,'o-','#e74c3c'), ('Random',rr,'s--','#95a5a6'), ('Frequency',rf,'^--','#f39c12'), ('Q-learned',rl,'D-','#3498db'), ('CEM',rc,'*-','#2ecc71'), ('Oracle',ro,'x-','#9b59b6')]
    for label, vals, marker, color in methods:
        if vals: ax.plot(Kr[:len(vals)], vals, marker, label=label, color=color, linewidth=1.5)
    ax.set_xlabel('Budget K'); ax.set_ylabel('MSE'); ax.set_title(f'{name}: Selection quality')
    ax.legend(); ax.grid(alpha=0.3)
    fig.savefig(os.path.join(output_dir, 'figures', f'mse_{name}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  [{(time.time()-t0):.0f}s] MSE curve saved')

    ai_q = np.mean([(rh[i]-rl[i])/max(rh[i],1e-10)*100 for i in range(len(Kr))])
    ai_cem = np.mean([(rh[i]-rc[i])/max(rh[i],1e-10)*100 for i in range(len(Kr))])
    ai_r = np.mean([(rh[i]-rr[i])/max(rh[i],1e-10)*100 for i in range(len(Kr))])
    ai_f = np.mean([(rh[i]-rf[i])/max(rh[i],1e-10)*100 for i in range(len(Kr))])
    print(f'  [{(time.time()-t0):.0f}s] Improv: Q={ai_q:.1f}% CEM={ai_cem:.1f}% Rand={ai_r:.1f}% Freq={ai_f:.1f}%')

    return {
        'scene':name,'N':N,'G':G,'T':T,
        'kappa_mean':kappa_mean,'pairs_coupled':pairs_coupled,'pct_coupled':pct_coupled,'total_pairs':total_pairs,
        'gate':overall_gate,'val_loss':val_loss,
        'avg_imp_q':ai_q,'avg_imp_cem':ai_cem,'avg_imp_rand':ai_r,'avg_imp_freq':ai_f,
        'regret_2':regs[0],'regret_3':regs[1],'regret_4':regs[2],
        'heuristic':rh,'q_learned':rl,'cem':rc,'oracle':ro,'random':rr,'freq':rf,
        'K_values':Kr,'runtime':time.time()-t0,'n_total_gaussians':n_total,'n_valid_gaussians':len(pos_v),
        'bootstrap_cis':cis,
    }

print('Pipeline function defined.')


In [ ]:
# Cell 4 - Multi-scene loop
print('\n' + '#'*70)
print('# MULTI-SCENE EVALUATION')
print('#'*70)

output_dir = os.path.join(os.getcwd(), '..')
results_all = {}
existing = {}
if os.path.exists(os.path.join(output_dir, 'results', 'results_all.json')):
    with open(os.path.join(output_dir, 'results', 'results_all.json')) as f: existing = json.load(f)
    print(f'Found {len(existing)} existing results')

for scene_name, scene_url in SCENES:
    if scene_name in existing:
        results_all[scene_name] = existing[scene_name]
        r = existing[scene_name]
        print(f'\nSKIP {scene_name}: already done')
        continue
    try:
        r = run_scene(scene_name, scene_url, output_dir)
        results_all[scene_name] = r
        with open(os.path.join(output_dir, 'results', 'results_all.json'), 'w') as f:
            json.dump(results_all, f, indent=1, cls=NpEncoder)
        print(f'  >> Saved results')
    except Exception as e:
        print(f'  >> FAILED: {e}')
        import traceback; traceback.print_exc()
        results_all[scene_name] = {'scene':scene_name,'error':str(e)}

n_ok = sum(1 for v in results_all.values() if 'error' not in v)
print(f'\nCompleted: {n_ok}/{len(SCENES)} scenes')


In [ ]:
# Cell 5 - Deep ablation study
print('\n' + '#'*70)
print('# DEEP ABLATION')
print('#'*70)

first = None
for v in results_all.values():
    if 'error' not in v: first = v; break
if first is None:
    print('No scene available. Run Cell 4 first.')
else:
    output_dir = os.path.join(os.getcwd(), '..')
    scene_name, scene_url = SCENES[0]
    print(f'Running deep ablation on {scene_name}...')
    t0 = time.time()

    pos_v, _, _, _ = download_scene(scene_url)
    w_base = VoxelWorld3D(N=80, G=16, T=120, pos_raw=pos_v)
    ov_base = overlap_iou(w_base)

    # Build full dataset
    X_base, y_base = [], []
    for t in range(w_base.T):
        b = w_base.base(t)
        for e in range(w_base.N): X_base.append(feats(w_base, t, e, ov_base)); y_base.append(b - w_base.upd([e], t)[0])
    X_base, y_base = np.array(X_base), np.array(y_base)

    # Feature definitions for ablation
    def feats_no_position(w,t,e,o):
        G=w.G; m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G; cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
        nv=np.sum(o[e]>.05)/(w.N-1); cl=np.zeros(5); cl[w.cid[e]%5]=1.
        return np.r_[m,[u,cc,ss,a,nv],cl]  # 13 feats (no position)
    def feats_no_motion(w,t,e,o):
        G=w.G; p=w.obs[t,e]/G
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G; cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
        nv=np.sum(o[e]>.05)/(w.N-1); cl=np.zeros(5); cl[w.cid[e]%5]=1.
        return np.r_[p,[u,cc,ss,a,nv],cl]  # 13 feats (no motion)
    def feats_no_uncertainty(w,t,e,o):
        G=w.G; p=w.obs[t,e]/G; m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
        cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
        nv=np.sum(o[e]>.05)/(w.N-1); cl=np.zeros(5); cl[w.cid[e]%5]=1.
        return np.r_[p,m,[cc,ss,a,nv],cl]  # 14 feats (no uncertainty)
    def feats_no_confidence(w,t,e,o):
        G=w.G; p=w.obs[t,e]/G; m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G; ss=w.sig[e]/G; a=w.amp[e]
        nv=np.sum(o[e]>.05)/(w.N-1); cl=np.zeros(5); cl[w.cid[e]%5]=1.
        return np.r_[p,m,[u,ss,a,nv],cl]  # 14 feats (no confidence)
    def feats_no_overlap(w,t,e,o):
        G=w.G; p=w.obs[t,e]/G; m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G; cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
        cl=np.zeros(5); cl[w.cid[e]%5]=1.
        return np.r_[p,m,[u,cc,ss,a],cl]  # 15 feats (no overlap)
    def feats_no_cluster(w,t,e,o):
        G=w.G; p=w.obs[t,e]/G; m=w.obs[t,e]-w.obs[t-1,e] if t>0 else np.zeros(3); m/=G
        u=np.linalg.norm(w.obs[t,e]-w.true[t,e])/G; cc=w.conf[t,e]; ss=w.sig[e]/G; a=w.amp[e]
        nv=np.sum(o[e]>.05)/(w.N-1)
        return np.r_[p,m,[u,cc,ss,a,nv]]  # 11 feats (no cluster)

    ablation_feats = {
        'no_position': feats_no_position,
        'no_motion': feats_no_motion,
        'no_uncertainty': feats_no_uncertainty,
        'no_confidence': feats_no_confidence,
        'no_overlap': feats_no_overlap,
        'no_cluster': feats_no_cluster,
    }

    # Train full model first
    mdl_full, sx_full, sy_full, vl_full = train_q(X_base, y_base, verbose=False)
    print(f'  [{(time.time()-t0):.0f}s] Full model: val_loss={vl_full:.4f}')

    # A. Feature removal ablation
    removal_results = {}
    for name, fn in ablation_feats.items():
        X2, y2 = [], []
        for t in range(w_base.T):
            b = w_base.base(t)
            for e in range(w_base.N): X2.append(fn(w_base, t, e, ov_base)); y2.append(b - w_base.upd([e], t)[0])
        _, _, _, vl = train_q(np.array(X2), np.array(y2), verbose=False)
        removal_results[name] = float(vl)
        print(f'  [{(time.time()-t0):.0f}s] {name}: val_loss={vl:.4f} (delta={vl-vl_full:+.4f})')

    # B. Permutation importance (no retrain needed)
    Xtr,Xva,ytr,yva=train_test_split(X_base,y_base,test_size=.2,random_state=42)
    sx_perm=StandardScaler().fit(Xtr); Xva_s=sx_perm.transform(Xva)
    perm_imp = {}
    feature_names = ['pos_x','pos_y','pos_z','mot_x','mot_y','mot_z','uncert','conf','scale','amp','overlap','cl0','cl1','cl2','cl3','cl4']
    with torch.no_grad():
        val_t = torch.FloatTensor(Xva_s)
        base_pred = mdl_full(val_t).squeeze().numpy()
        base_mse = np.mean((base_pred - yva)**2)
        for fi in range(16):
            X_perm = Xva_s.copy()
            np.random.shuffle(X_perm[:, fi])
            perm_pred = mdl_full(torch.FloatTensor(X_perm)).squeeze().numpy()
            perm_mse = np.mean((perm_pred - yva)**2)
            perm_imp[feature_names[fi]] = float(perm_mse - base_mse)
    print(f'  [{(time.time()-t0):.0f}s] Permutation importance computed')

    # C. Varying N
    variants_N = {}
    for N_test in [40, 80, 120]:
        w_n = VoxelWorld3D(N=N_test, G=16, T=120, pos_raw=pos_v)
        ov_n = overlap_iou(w_n)
        Xn, yn = [], []
        for t in range(w_n.T):
            b = w_n.base(t)
            for e in range(w_n.N): Xn.append(feats(w_n, t, e, ov_n)); yn.append(b - w_n.upd([e], t)[0])
        _, _, _, vl_n = train_q(np.array(Xn), np.array(yn), verbose=False)
        variants_N[str(N_test)] = float(vl_n)
        print(f'  [{(time.time()-t0):.0f}s] N={N_test}: val_loss={vl_n:.4f}')

    # D. No dropout
    _, _, _, vl_drop = train_q(X_base, y_base, dropout=False, verbose=False)
    print(f'  [{(time.time()-t0):.0f}s] No dropout: val_loss={vl_drop:.4f} (delta={vl_drop-vl_full:+.4f})')

    ablation = {
        'full': vl_full,
        'feature_removal': removal_results,
        'permutation_importance': perm_imp,
        'varying_N': variants_N,
        'no_dropout': float(vl_drop),
    }
    with open(os.path.join(output_dir, 'results', 'ablation.json'), 'w') as f:
        json.dump(ablation, f, indent=1, cls=NpEncoder)
    print(f'  [{(time.time()-t0):.0f}s] Ablation saved')

    # Print summary
    print('\nFeature removal ranking (worst first):')
    ranked = sorted(removal_results.items(), key=lambda x: x[1]-vl_full, reverse=True)
    for name, vl in ranked:
        print(f'  {name:20s}: delta={vl-vl_full:+.4f}')
    print('\nPermutation importance (worst first):')
    ranked_p = sorted(perm_imp.items(), key=lambda x: x[1], reverse=True)
    for name, imp in ranked_p:
        print(f'  {name:20s}: increase={imp:.4f}')
    print(f'\nVarying N: {variants_N}')
    print(f'No dropout: {vl_drop:.4f} (full={vl_full:.4f})')


In [ ]:
# Cell 6 - Final comparison table + LaTeX output
print('\n' + '#'*70)
print('# FINAL RESULTS')
print('#'*70)

output_dir = os.path.join(os.getcwd(), '..')
valid = {k:v for k,v in results_all.items() if 'error' not in v}

print('\n' + '-'*70)
print(f'{"Scene":>12} {"|k|":>7} {"Coup%":>7} {"Q-%":>7} {"CEM%":>7} {"Rnd%":>7} {"Freq%":>7} {"Q>H":>5} {"CEM>H":>5}')
print('-'*70)
sav = {'kappa':[],'q':[],'cem':[],'rnd':[],'freq':[]}
for name, r in sorted(valid.items()):
    qf = r['avg_imp_q']; cf = r['avg_imp_cem']; rf = r['avg_imp_rand']; ff = r['avg_imp_freq']
    print(f'{r["scene"]:>12} {r["kappa_mean"]:7.4f} {r["pct_coupled"]:6.1f}% {qf:+6.1f}% {cf:+6.1f}% {rf:+6.1f}% {ff:+6.1f}% {"OK" if qf>3 else "":>5} {"OK" if cf>3 else "":>5}')
    for k,v in [('kappa',r['kappa_mean']),('q',qf),('cem',cf),('rnd',rf),('freq',ff)]: sav[k].append(v)
if valid:
    print('-'*70)
    print(f'{"AVERAGE":>12} {np.mean(sav["kappa"]):7.4f}        {np.mean(sav["q"]):+6.1f}% {np.mean(sav["cem"]):+6.1f}% {np.mean(sav["rnd"]):+6.1f}% {np.mean(sav["freq"]):+6.1f}%')
    print('-'*70)
print(f'\n{len(valid)}/{len(SCENES)} scenes')

# Per-K breakdown
print('\n' + '-'*60)
print('Per-K breakdown (avg across scenes):')
print(f'{"K":>3} {"Heuristic":>10} {"Q":>8} {"CEM":>8} {"Rand":>8} {"Freq":>8} {"Q-H%":>7} {"CEM-H%":>7}')
print('-'*60)
for Ki in range(8):
    h,q,c,r,f=[],[],[],[],[]
    for v2 in valid.values():
        if Ki<len(v2['heuristic']): h.append(v2['heuristic'][Ki]); q.append(v2['q_learned'][Ki]); c.append(v2['cem'][Ki]); r.append(v2['random'][Ki]); f.append(v2['freq'][Ki])
    if h:
        hm,qm,cm,rm,fm=np.mean(h),np.mean(q),np.mean(c),np.mean(r),np.mean(f)
        print(f'{Ki+1:3d} {hm:10.4f} {qm:8.4f} {cm:8.4f} {rm:8.4f} {fm:8.4f} {(hm-qm)/max(hm,1e-10)*100:+6.1f}% {(hm-cm)/max(hm,1e-10)*100:+6.1f}%')

# Auto-generate LaTeX table
latex = []
'% Auto-generated LaTeX table - do not edit manually'
'\\begin{table*}[t]'
'\\centering'
'\\caption{Multi-scene results across ' + str(len(valid)) + ' real 3DGS scenes. Methods: H (heuristic), R (random), F (frequency), Q (learned), CEM. Oracle at K<=4.}'
'\\label{tab:final}'
'\\begin{tabular}{lrrrrrrr}'
'\\toprule'
'Scene & |k| & Coupled% & H MSE & R MSE & F MSE & Q MSE & CEM MSE \\\\'
'\\midrule'for name, r in sorted(valid.items()):
    hm = r['heuristic'][3]; qm = r['q_learned'][3]; cm = r['cem'][3]; rm = r['random'][3]; fm = r['freq'][3]
    latex.append(f'{r["scene"]} & {r["kappa_mean"]:.4f} & {r["pct_coupled"]:.0f}\\\% & {hm:.4f} & {rm:.4f} & {fm:.4f} & {qm:.4f} & {cm:.4f} \\\\')
latex.append('\\midrule')
latex.append(f'Avg & {np.mean(sav["kappa"]):.4f} & -- & -- & -- & -- & {np.mean(sav["q"]):+.1f}\\\% & {np.mean(sav["cem"]):+.1f}\\\% \\\\')
latex.append('\\bottomrule')
latex.append('\\end{tabular}')
latex.append('\\end{table*}')
latex_str = chr(10).join(latex)
with open(os.path.join(output_dir, 'results', 'latex_table.tex'), 'w') as f: f.write(latex_str)
print(f'\nLaTeX table saved to results/latex_table.tex')

fig, axes = plt.subplots(2, 3, figsize=(12,7))
axes = axes.flatten()
for idx, (name, r) in enumerate(sorted(valid.items())):
    if idx >= 6: break
    ax = axes[idx]
    Ks = r['K_values']
    ax.plot(Ks, r['heuristic'], 'o-', color='#e74c3c', label='H', lw=1.5)
    ax.plot(Ks, r['random'], 's--', color='#95a5a6', label='R', lw=1.5)
    ax.plot(Ks, r['freq'], '^--', color='#f39c12', label='F', lw=1.5)
    ax.plot(Ks, r['q_learned'], 'D-', color='#3498db', label='Q', lw=1.5)
    ax.plot(Ks, r['cem'], '*-', color='#2ecc71', label='CEM', lw=1.5)
    if r.get('oracle'): ax.plot(Ks[:len(r['oracle'])], r['oracle'], 'x-', color='#9b59b6', label='O', lw=1.5)
    ax.set_title(f'{name}', fontsize=10)
    ax.set_xlabel('K'); ax.set_ylabel('MSE')
    ax.grid(alpha=0.2); ax.legend(fontsize=7)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, 'figures', 'all_scenes_comparison.png'), dpi=150, bbox_inches='tight')
plt.close(fig)
print('Comparison figure saved to figures/all_scenes_comparison.png')


In [ ]:
# Cell 7 - Save aggregated summary + paper-ready numbers
valid = {k:v for k,v in results_all.items() if 'error' not in v}
output_dir = os.path.join(os.getcwd(), '..')

summary = {
    'n_scenes': len(valid),
    'scenes': list(valid.keys()),
    'kappa_mean_avg': float(np.mean([r['kappa_mean'] for r in valid.values()])),
    'kappa_mean_std': float(np.std([r['kappa_mean'] for r in valid.values()])),
    'q_imp_avg': float(np.mean([r['avg_imp_q'] for r in valid.values()])),
    'q_imp_std': float(np.std([r['avg_imp_q'] for r in valid.values()])),
    'cem_imp_avg': float(np.mean([r['avg_imp_cem'] for r in valid.values()])),
    'cem_imp_std': float(np.std([r['avg_imp_cem'] for r in valid.values()])),
    'rand_imp_avg': float(np.mean([r['avg_imp_rand'] for r in valid.values()])),
    'freq_imp_avg': float(np.mean([r['avg_imp_freq'] for r in valid.values()])),
    'avg_val_loss': float(np.mean([r['val_loss'] for r in valid.values()])),
    'per_scene': valid,
}

ablation = {}
if os.path.exists(os.path.join(output_dir, 'results', 'ablation.json')):
    with open(os.path.join(output_dir, 'results', 'ablation.json')) as f: ablation = json.load(f)
summary['ablation'] = ablation
summary['version'] = 'v2.0-multiscene-cem-ablation'

with open(os.path.join(output_dir, 'results', 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=1, cls=NpEncoder)

print('Summary saved to results/summary.json')

# Paper-ready numbers
kappa_all = [r['kappa_mean'] for r in valid.values()]
q_all = [r['avg_imp_q'] for r in valid.values()]
cem_all = [r['avg_imp_cem'] for r in valid.values()]
rnd_all = [r['avg_imp_rand'] for r in valid.values()]
freq_all = [r['avg_imp_freq'] for r in valid.values()]
print(f'\n{"="*60}')
print('PAPER-READY SUMMARY')
print(f'{"="*60}')
print(f'  Multi-scene ({len(valid)} real 3DGS scenes):')
print(f'    |k| = {np.mean(kappa_all):.4f} +/- {np.std(kappa_all):.4f} (range {min(kappa_all):.4f}-{max(kappa_all):.4f})')
print(f'    Q improvement:  {np.mean(q_all):.1f}% +/- {np.std(q_all):.1f}%  (range {min(q_all):.1f}%-{max(q_all):.1f}%)')
print(f'    CEM improvement: {np.mean(cem_all):.1f}% +/- {np.std(cem_all):.1f}%')
print(f'    Random improvement: {np.mean(rnd_all):.1f}%')
print(f'    Frequency improvement: {np.mean(freq_all):.1f}%')
print(f'  Q > H in {sum(1 for r in valid.values() if r["avg_imp_q"]>3)}/{len(valid)} scenes')
print(f'  CEM > H in {sum(1 for r in valid.values() if r["avg_imp_cem"]>3)}/{len(valid)} scenes')
print(f'  CEM > Q in {sum(1 for r in valid.values() if r["avg_imp_cem"]>r["avg_imp_q"])}/{len(valid)} scenes (+{np.mean(cem_all)-np.mean(q_all):.1f}% avg)')
print(f'  {"="*60}')
print(f'Generated files:')
print(f'  figures/kappa_*.png         - 6 coupling heatmaps')
print(f'  figures/mse_*.png            - 6 per-scene MSE curves')
print(f'  figures/all_scenes_comparison.png - aggregated 6-panel figure')
print(f'  results/results_all.json     - per-scene raw results')
print(f'  results/ablation.json        - deep ablation results')
print(f'  results/latex_table.tex      - auto-generated LaTeX table')
print(f'  results/summary.json         - aggregated summary')
